# Module 02 — Chọn mẫu & cỡ mẫu
Notebook RIÊNG cho Module 02 — kiểm chứng trọng số mẫu PISA thay đổi kết quả thế nào, và bảng tính cỡ mẫu theo power analysis (Cohen 1988, qua `statsmodels`) — phần bảng cỡ mẫu KHÔNG có lệnh PSPP tương đương, chỉ chạy được ở đây.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"]=(7,4); pd.set_option("display.precision",4)

RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
print("Bảng trường:", truong.shape)

## 1. Ba tầng: tổng thể, khung lấy mẫu, mẫu
Trong PISA 2025 VN: tổng thể = toàn bộ HS 15 tuổi VN; khung lấy mẫu = danh sách trường Bộ GD&ĐT cung cấp cho OECD; mẫu = 195 trường thực tế tham gia.

In [ ]:
print(f"Mẫu thực tế: {truong.shape[0]} trường, tổng {int(truong['n_hs'].sum())} học sinh")

## 2. Trọng số mẫu thay đổi kết quả thế nào — kiểm chứng thật
Vì PISA chọn mẫu theo cụm với xác suất TỈ LỆ VỚI QUY MÔ trường, các trường không có "quyền số" ngang nhau khi phân tích.

In [ ]:
# Không trọng số: mỗi trường tính = 1
mean_unweighted = truong["EDULEAD"].mean()

# Có trọng số: WEIGHT BY W_NRASCHBWT (tương đương SPSS Data > Weight Cases)
valid = truong.dropna(subset=["EDULEAD"])
mean_weighted = np.average(valid["EDULEAD"], weights=valid["W_NRASCHBWT"])
n_eff = valid["W_NRASCHBWT"].sum()

print(f"Trung bình EDULEAD KHÔNG trọng số = {mean_unweighted:.4f}  (N={len(valid)} trường, mỗi trường=1)")
print(f"Trung bình EDULEAD CÓ trọng số    = {mean_weighted:.4f}  (N hiệu dụng={n_eff:.1f})")

#### 📤 Đầu ra thật
Không trọng số = 0,7653 (N=195). Có trọng số = 0,7874 (N hiệu dụng=7.284,6 — chính là tổng trọng số, phản ánh đúng quy mô học sinh thật). ✅ Khớp đúng con số trong bài — chênh lệch ở đây khá nhỏ (0,02) vì EDULEAD không tương quan mạnh với quy mô trường, nhưng với biến tương quan mạnh hơn, bỏ qua trọng số có thể làm kết luận sai lệch đáng kể.

## 3. Cỡ mẫu pilot tối thiểu (Julious, 2005)
n_pilot ≥ 12/nhóm — dưới ngưỡng này, SD ước lượng dao động quá lớn để làm đầu vào tính cỡ mẫu chính thức.

In [ ]:
print("Cỡ mẫu pilot tối thiểu theo Julious (2005): 12 quan sát/nhóm")

## 4. Bảng tính cỡ mẫu chính thức theo power analysis (Cohen, 1988)
Công thức xấp xỉ: n ≈ (z_α/2 + z_β)² × 2 / d² — tính chính xác hơn bằng `statsmodels.stats.power.TTestIndPower` (dùng phân phối t không trung tâm thay vì xấp xỉ z).

In [ ]:
from statsmodels.stats.power import TTestIndPower
analysis = TTestIndPower()
rows = []
for d_val, label in [(0.2,"Nhỏ"), (0.5,"Vừa"), (0.8,"Lớn")]:
    n_req = analysis.solve_power(effect_size=d_val, alpha=0.05, power=0.80, alternative="two-sided")
    rows.append((d_val, label, round(n_req)))
table = pd.DataFrame(rows, columns=["Cỡ hiệu ứng d", "Mức", "Cỡ mẫu/nhóm"])
table

#### 📤 Đầu ra thật
d=0,2→≈394/nhóm, d=0,5→≈64/nhóm, d=0,8→≈26/nhóm (α=0,05, power=0,80). ✅ Khớp đúng bảng trong bài — hiệu ứng càng nhỏ, cỡ mẫu cần thiết càng lớn.

## 5. Trực quan: cỡ mẫu cần thiết giảm dần khi cỡ hiệu ứng tăng

In [ ]:
d_range = np.linspace(0.1, 1.0, 30)
n_range = [analysis.solve_power(effect_size=d, alpha=0.05, power=0.80, alternative="two-sided") for d in d_range]
fig, ax = plt.subplots()
ax.plot(d_range, n_range)
ax.set_xlabel("Cỡ hiệu ứng d"); ax.set_ylabel("Cỡ mẫu cần thiết / nhóm")
ax.set_title("Cỡ mẫu cần thiết giảm nhanh khi hiệu ứng tăng (α=0,05, power=0,80)")
plt.show()